In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import butter, sosfilt, spectrogram
from IPython.display import Audio
import soundfile as sf
import torch

# ----------------------------
# Paths
# ----------------------------
noisy_file = r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Bluetit_Prediction_Test\Bluetit_test3.wav"  
output_folder = r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Denoised_Bluetit_Prediction"
os.makedirs(output_folder, exist_ok=True)

# ----------------------------
# Load the noisy audio
# ----------------------------
noisy_audio, sr = librosa.load(noisy_file, sr=None)

# ----------------------------
# Convert to STFT
# ----------------------------
n_fft = 1024
hop_length = 512
noisy_stft = librosa.stft(noisy_audio, n_fft=n_fft, hop_length=hop_length)
noisy_mag = np.abs(noisy_stft)
noisy_phase = np.angle(noisy_stft)

# ----------------------------
# Convert to PyTorch Tensor
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
noisy_mag_tensor = torch.tensor(noisy_mag.T, dtype=torch.float32).to(device)

# ----------------------------
# Load Trained Model
# ----------------------------
class EnhancedDenoiser(torch.nn.Module):
    def __init__(self, input_dim):
        super(EnhancedDenoiser, self).__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 1024), torch.nn.ReLU(), torch.nn.BatchNorm1d(1024),
            torch.nn.Linear(1024, 512), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(512, 256), torch.nn.ReLU(), torch.nn.BatchNorm1d(256),
            torch.nn.Linear(256, input_dim), torch.nn.ReLU()
        )
    def forward(self, x): return self.net(x)

# Load model
input_dim = n_fft // 2 + 1
model = EnhancedDenoiser(input_dim).to(device)
model.load_state_dict(torch.load(r"C:\Users\benji\Documents\Bird_Project_Code\Filter_Bluetit\Bluetit_FNN_Filter_Trained.pth"))  # Change to your saved model path
model.eval()

# ----------------------------
# Denoise the Audio
# ----------------------------
with torch.no_grad():
    denoised_mag = model(noisy_mag_tensor).cpu().numpy().T  # Predict clean magnitude

# Restore STFT
denoised_stft = denoised_mag * np.exp(1j * noisy_phase)  # Keep original phase

# Convert back to time domain
denoised_audio = librosa.istft(denoised_stft, hop_length=hop_length)

# ----------------------------
# Save the denoised audio
# ----------------------------
output_path = os.path.join(output_folder, "Denoised_Test_3.wav")
sf.write(output_path, denoised_audio, sr)

print(f"Denoised audio saved to: {output_path}")


# Constants
SOUND_SPEED = 343.2  # Speed of sound in air (m/s)
STOPBAND_FACTOR = 1.5  # Chosen multiplier to define the stopband frequency

# Define microphone pairs with unique positions and LPF parameters.
mic_pairs = {
    "mic_pair_1": {
        "positions": [np.array([-0.06, 0]), np.array([0.06, 0])],
        "lpf_params": {"cutoff": 250, "A_p": 1, "A_s": 20}  # 343.2/250 ≈ 1.37 m > 0.12 m separation
    },
    "mic_pair_2": {
        "positions": [np.array([-0.12, 0]), np.array([0.12, 0])],
        "lpf_params": {"cutoff": 400, "A_p": 1, "A_s": 25}  # 343.2/400 ≈ 0.86 m > 0.24 m separation
    },
    "mic_pair_3": {
        "positions": [np.array([-1, 0]), np.array([1, 0])], # 343.2/150 ≈ 2.29 m > 2 m separation.
        "lpf_params": {"cutoff": 150, "A_p": 1, "A_s": 30}
    }
}

# Function to calculate Butterworth filter order
def butterworth_order(A_p, A_s, omega_p, omega_s):
    return int(np.ceil(np.log10((10**(A_s/10) - 1) / (10**(A_p/10) - 1)) / (2 * np.log10(omega_s / omega_p))))

# Load audio file
folder_path = r'Denoised_Bluetit_Prediction'
filename = "Denoised_Test_3.wav"  
file_path = os.path.join(folder_path, filename)
SR, y = wavfile.read(file_path)

# Display the audio 
display(Audio(y, rate=SR))

# Filter order for each microphone pair using LPF parameters
for pair_name, pair in mic_pairs.items():
    cutoff = pair["lpf_params"]["cutoff"]
    A_p = pair["lpf_params"]["A_p"]
    A_s = pair["lpf_params"]["A_s"]
    omega_p = cutoff / (SR / 2)
    omega_s = cutoff * STOPBAND_FACTOR / (SR / 2)
    order = butterworth_order(A_p, A_s, omega_p, omega_s)
    pair["lpf_params"]["order"] = order
    print(f"{pair_name}: LPF order = {order}")

# LPF function using second-order sections
def apply_lpf(signal_data, cutoff, order, fs):
    sos = butter(order, cutoff, btype='lp', fs=fs, output='sos')
    return sosfilt(sos, signal_data)

# Function to rotate a microphone pair about its own midpoint by a given angle
def rotate_mic_pair(angle, mic_positions):
    midpoint = (mic_positions[0] + mic_positions[1]) / 2
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    return [midpoint + R @ (pos - midpoint) for pos in mic_positions]

# Function to compute loss for a given angle, sound source, and microphone pair.
def loss_angle_with_lpf_for_pair(angle, sound_source, mic_pair, SR):
    mic_positions = mic_pair["positions"]
    lpf_params = mic_pair["lpf_params"]
    rotated_mics = rotate_mic_pair(angle, mic_positions)
    mic1, mic2 = rotated_mics[0], rotated_mics[1]
    dist1 = np.linalg.norm(sound_source - mic1)
    dist2 = np.linalg.norm(sound_source - mic2)
    t1 = dist1 / SOUND_SPEED
    t2 = dist2 / SOUND_SPEED
    samples_delay1 = int(t1 * SR)
    samples_delay2 = int(t2 * SR)
    y1 = y[samples_delay1:]
    y2 = y[samples_delay2:]
    min_len = min(len(y1), len(y2))
    y1 = y1[:min_len]
    y2 = y2[:min_len]
    y1_filtered = apply_lpf(y1, lpf_params["cutoff"], lpf_params["order"], SR)
    y2_filtered = apply_lpf(y2, lpf_params["cutoff"], lpf_params["order"], SR)
    return np.linalg.norm(y1_filtered - y2_filtered)

# Define the sound source position
sound_source = np.array([1, 3])

# Define angles (in radians) for microphone rotation
angles = np.linspace(-np.pi/2, np.pi/2, 501)

# Plot losses for each microphone pair
plt.figure(figsize=(10, 6))
for pair_name, pair in mic_pairs.items():
    losses = np.array([loss_angle_with_lpf_for_pair(a, sound_source, pair, SR) for a in angles])
    plt.plot(np.degrees(angles), losses, label=f"{pair_name}", linestyle=':')
plt.xlabel("Rotation Angle (degrees)")
plt.ylabel("Loss (L2 norm)")
plt.title("Loss vs Rotation Angle for each Microphone Pair")
plt.legend()
plt.grid(True)
plt.show()

# Compute and plot the overall combined loss
overall_loss = np.zeros_like(angles)
for pair_name, pair in mic_pairs.items():
    losses = np.array([loss_angle_with_lpf_for_pair(a, sound_source, pair, SR) for a in angles])
    overall_loss += losses

plt.figure(figsize=(10, 6))
plt.plot(np.degrees(angles), overall_loss, label="Overall Combined Loss", color='black', linewidth=2)
plt.xlabel("Rotation Angle (degrees)")
plt.ylabel("Loss (L2 norm)")
plt.title("Overall Combined Loss vs Rotation Angle")
plt.legend()
plt.grid(True)
plt.show()